In [ ]:
from datasets import load_dataset

dataset = load_dataset("facebook/anli", split='train_r1')
dataset

# 0 entailment
# 1 neutral
# 2 contradiction
df = dataset.to_pandas()
df.head()

In [23]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import yaml

PROJECT_ROOT = Path('/Users/nident/Desktop/JOB/startup/agent_scorer')
OUTPUT_DIR = PROJECT_ROOT / 'evaluation_ANLI' / 'generated'
DIALOGUES_DIR = OUTPUT_DIR / 'dialogues'
CRITERIA_DIR = OUTPUT_DIR / 'criteria'
INDEX_PATH = OUTPUT_DIR / 'index.jsonl'

LABEL_ID_TO_NAME = {
    0: 'entailment',
    1: 'neutral',
    2: 'contradiction',
}

MAX_GROUPS = None  # set an int, for example 10, if you want to test on a small sample


def label_name(label):
    if label in LABEL_ID_TO_NAME:
        return LABEL_ID_TO_NAME[label]
    return LABEL_ID_TO_NAME.get(int(label), str(label))


def safe_filename(value):
    value = re.sub(r'[^a-zA-Z0-9_-]+', '_', str(value)).strip('_')
    return value[:120] or 'item'


DIALOGUES_DIR.mkdir(parents=True, exist_ok=True)
CRITERIA_DIR.mkdir(parents=True, exist_ok=True)

created = []

for premise_index, (premise, group_df) in enumerate(df.groupby('premise', sort=False), start=1):
    if MAX_GROUPS is not None and premise_index > MAX_GROUPS:
        break

    group_df = group_df.reset_index(drop=True)
    first_uid = str(group_df.loc[0, 'uid'])
    record_id = f'premise_{premise_index:06d}_{safe_filename(first_uid)}'
    created_at = datetime.now(timezone.utc).isoformat()

    dialogue_path = DIALOGUES_DIR / f'{record_id}.json'
    criterion_path = CRITERIA_DIR / f'{record_id}.yaml'

    dialogue = {
        'schema_version': 'transcript_json_v1',
        'created_at': created_at,
        'transcript': {
            'source_format': 'speaker_turns',
            'source_file': str(dialogue_path.relative_to(PROJECT_ROOT)),
            'source_dataset': 'facebook/anli',
            'language': 'en',
            'turn_count': 2,
            'speakers': [
                {'speaker_label': 'A', 'role': 'unknown', 'utterance_count': 1},
                {'speaker_label': 'B', 'role': 'unknown', 'utterance_count': 1},
            ],
            'utterances': [
                {
                    'timestamp': None,
                    'speaker_label': 'A',
                    'role': 'unknown',
                    'text': '',
                    'turn_index': 1,
                },
                {
                    'timestamp': None,
                    'speaker_label': 'B',
                    'role': 'unknown',
                    'text': str(premise),
                    'turn_index': 2,
                },
            ],
        },
    }

    gold_items = [
        {
            'uid': str(row.uid),
            'label_id': int(row.label),
            'label': label_name(row.label),
            'hypothesis': str(row.hypothesis),
        }
        for row in group_df.itertuples(index=False)
    ]

    criterion = {
        'name': f'ANLI hypotheses for {record_id}',
        'premise_id': record_id,
        'points': [
            f"{item['hypothesis']}"
            for item in gold_items
        ],
        'gold': gold_items,
    }

    dialogue_path.write_text(json.dumps(dialogue, ensure_ascii=False, indent=2), encoding='utf-8')
    criterion_path.write_text(
        yaml.safe_dump(criterion, allow_unicode=True, sort_keys=False),
        encoding='utf-8',
    )

    created.append(
        {
            'premise_id': record_id,
            'dialogue_path': str(dialogue_path.relative_to(PROJECT_ROOT)),
            'criterion_path': str(criterion_path.relative_to(PROJECT_ROOT)),
            'hypothesis_count': len(gold_items),
            'uids': [item['uid'] for item in gold_items],
        }
    )

with INDEX_PATH.open('w', encoding='utf-8') as file:
    for item in created:
        file.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Created {len(created)} premise groups')
print(f'Dialogues: {DIALOGUES_DIR}')
print(f'Criteria:  {CRITERIA_DIR}')
print(f'Index:     {INDEX_PATH}')

Created 2054 premise groups
Dialogues: /Users/nident/Desktop/JOB/startup/agent_scorer/evaluation_ANLI/generated/dialogues
Criteria:  /Users/nident/Desktop/JOB/startup/agent_scorer/evaluation_ANLI/generated/criteria
Index:     /Users/nident/Desktop/JOB/startup/agent_scorer/evaluation_ANLI/generated/index.jsonl
